# 1\.2\.1 Standardize Temporal Representations

This notebook creates the shared temporal fields that each mobility dataset will need before spatial alignment and multimodal integration\. The goal is to preserve each dataset's useful time structure while adding consistent fields for date, weekday, hour, temporal bucket, and congestion\-pricing period\. These standardized outputs will remain modality\-specific and will feed into 1\.2\.2 for spatial standardization\.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

import duckdb  # For those huge parquet files!
import gc

📝 Note: Across modalities, this notebook is standardizing toward the same core temporal fields: date where available, year, month, day\_of\_week, hour, temporal\_bucket, and pre\_post\_cp\. Some source datasets may not support every field—for example, the bus speed data does not include a true calendar date—but the goal is to keep the shared temporal representation as consistent as possible\. Extra analytical fields like daypart, trip duration, wait time, distance, or speed should be handled later during feature engineering rather than added here\.

In [2]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

TRAFFIC_COUNTS_STAGED_PATH = PIPELINE_DATA_DIR / "1.1.1.traffic_counts_staged.parquet"
TRAFFIC_COUNTS_TEMPORAL_PATH = PIPELINE_DATA_DIR / "1.2.1.traffic_counts_temporal.parquet"

## 1\.2\.1\.1 Add temporal fields to traffic data

In [3]:
# ---------------------------------------------------------------------
# Load staged traffic counts
# ---------------------------------------------------------------------

traffic_counts = pd.read_parquet(TRAFFIC_COUNTS_STAGED_PATH)

traffic_counts.head()

,segment_id,borough,street_name,from_street,to_street,direction,timestamp,traffic_volume,geometry_wkt,observation_count
0,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:00:00,16,POINT (932655.1292010084 168977.97581364735),1
1,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:15:00,19,POINT (932655.1292010084 168977.97581364735),1
2,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:30:00,10,POINT (932655.1292010084 168977.97581364735),1
3,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:45:00,12,POINT (932655.1292010084 168977.97581364735),1
4,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 01:00:00,16,POINT (932655.1292010084 168977.97581364735),1


In [4]:
# ---------------------------------------------------------------------
# Temporal standardization helpers
# ---------------------------------------------------------------------

CONGESTION_PRICING_START_DATE = pd.Timestamp("2025-01-05")

def assign_temporal_bucket(day_of_week, hour):
    day_type = "weekend" if day_of_week in ["Saturday", "Sunday"] else "weekday"

    if 0 <= hour <= 5:
        daypart = "overnight"
    elif 6 <= hour <= 9:
        daypart = "am_peak"
    elif 10 <= hour <= 15:
        daypart = "midday"
    elif 16 <= hour <= 19:
        daypart = "pm_peak"
    elif 20 <= hour <= 23:
        daypart = "evening"
    else:
        return np.nan

    return f"{day_type}_{daypart}"

In [5]:
# ---------------------------------------------------------------------
# Add standardized temporal fields to traffic counts
# ---------------------------------------------------------------------

traffic_temporal = traffic_counts.copy()

traffic_temporal["timestamp"] = pd.to_datetime(traffic_temporal["timestamp"])

traffic_temporal["date"] = traffic_temporal["timestamp"].dt.date
traffic_temporal["year"] = traffic_temporal["timestamp"].dt.year
traffic_temporal["month"] = traffic_temporal["timestamp"].dt.month
traffic_temporal["day_of_week"] = traffic_temporal["timestamp"].dt.day_name()
traffic_temporal["hour"] = traffic_temporal["timestamp"].dt.hour

traffic_temporal["temporal_bucket"] = traffic_temporal.apply(
    lambda row: assign_temporal_bucket(row["day_of_week"], row["hour"]),
    axis=1
)

traffic_temporal["pre_post_cp"] = np.where(
    traffic_temporal["timestamp"] >= CONGESTION_PRICING_START_DATE,
    "post",
    "pre"
)

traffic_temporal.head()

,segment_id,borough,street_name,from_street,to_street,direction,timestamp,traffic_volume,geometry_wkt,observation_count,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp
0,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:00:00,16,POINT (932655.1292010084 168977.97581364735),1,2023-09-06,2023,9,Wednesday,0,weekday_overnight,pre
1,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:15:00,19,POINT (932655.1292010084 168977.97581364735),1,2023-09-06,2023,9,Wednesday,0,weekday_overnight,pre
2,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:30:00,10,POINT (932655.1292010084 168977.97581364735),1,2023-09-06,2023,9,Wednesday,0,weekday_overnight,pre
3,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:45:00,12,POINT (932655.1292010084 168977.97581364735),1,2023-09-06,2023,9,Wednesday,0,weekday_overnight,pre
4,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 01:00:00,16,POINT (932655.1292010084 168977.97581364735),1,2023-09-06,2023,9,Wednesday,1,weekday_overnight,pre


In [6]:
# ---------------------------------------------------------------------
# Validate standardized temporal fields
# ---------------------------------------------------------------------

temporal_fields = [
    "timestamp",
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "temporal_bucket",
    "pre_post_cp",
]

temporal_qa = pd.DataFrame({
    "field": temporal_fields,
    "missing_count": [traffic_temporal[col].isna().sum() for col in temporal_fields],
    "missing_rate": [traffic_temporal[col].isna().mean() for col in temporal_fields],
    "unique_values": [traffic_temporal[col].nunique(dropna=True) for col in temporal_fields],
})

display(temporal_qa)

print("Temporal bucket distribution:")
display(traffic_temporal["temporal_bucket"].value_counts(dropna=False).sort_index())

print("Pre/post congestion pricing distribution:")
display(traffic_temporal["pre_post_cp"].value_counts(dropna=False))

,field,missing_count,missing_rate,unique_values
0,timestamp,0,0.0,68452
1,date,0,0.0,714
2,year,0,0.0,4
3,month,0,0.0,12
4,day_of_week,0,0.0,7
5,hour,0,0.0,24
6,temporal_bucket,0,0.0,10
7,pre_post_cp,0,0.0,2


Temporal bucket distribution:


temporal_bucket
weekday_am_peak      31214
weekday_evening      31201
weekday_midday       46823
weekday_overnight    46820
weekday_pm_peak      31215
weekend_am_peak      13231
weekend_evening      13245
weekend_midday       19848
weekend_overnight    19821
weekend_pm_peak      13233
Name: count, dtype: int64

Pre/post congestion pricing distribution:


pre_post_cp
pre     175163
post     91488
Name: count, dtype: int64

Findings\. Temporal standardization completed cleanly for the traffic dataset\. All standardized temporal fields were populated without missing values, with full coverage across dates, hours, weekdays, temporal buckets, and pre/post congestion\-pricing periods\.

In [7]:
# ---------------------------------------------------------------------
# Write temporal-standardized traffic counts
# ---------------------------------------------------------------------

traffic_temporal.to_parquet(
    TRAFFIC_COUNTS_TEMPORAL_PATH,
    index=False
)

print(f"Wrote temporal-standardized traffic counts to: {TRAFFIC_COUNTS_TEMPORAL_PATH}")
print(f"Rows written: {len(traffic_temporal):,}")

Wrote temporal-standardized traffic counts to: pipeline_data/1.2.1.traffic_counts_temporal.parquet
Rows written: 266,651


## 1\.2\.1\.2 Add temporal fields to bus data

This section standardizes the temporal representation of the bus dataset using the Year, Month, Day of Week, and Hour of Day fields identified during dataset inspection\. Unlike the traffic dataset, the Timestamp field does not appear to function as a reliable temporal identifier \(at least for hour \-\- the date seems reliable\), so temporal standardization will rely on the individual temporal fields instead\.

In [8]:
# ---------------------------------------------------------------------
# Configure bus temporal standardization paths
# ---------------------------------------------------------------------

BUS_STAGED_DIR = PIPELINE_DATA_DIR / "1.1.4 bus_route_segment_speeds_parquet"
BUS_TEMPORAL_DIR = PIPELINE_DATA_DIR / "1.2.1.bus_speeds_temporal_parquet"

BUS_TEMPORAL_DIR.mkdir(parents=True, exist_ok=True)

bus_staged_files = sorted(BUS_STAGED_DIR.glob("*.parquet"))

print(f"Found {len(bus_staged_files)} staged bus parquet files.")
print(f"Temporal-standardized bus output directory: {BUS_TEMPORAL_DIR}")

Found 39 staged bus parquet files.
Temporal-standardized bus output directory: pipeline_data/1.2.1.bus_speeds_temporal_parquet


In [9]:
# ---------------------------------------------------------------------
# Add standardized temporal fields to bus parquet partitions
# ---------------------------------------------------------------------

REBUILD_BUS_TEMPORAL = False

if any(BUS_TEMPORAL_DIR.glob("*.parquet")) and not REBUILD_BUS_TEMPORAL:
    print("Temporal-standardized bus files already exist.")
    print("Set REBUILD_BUS_TEMPORAL = True to regenerate them.")

else:
    for input_path in bus_staged_files:
        output_path = BUS_TEMPORAL_DIR / input_path.name

        df = pd.read_parquet(input_path)

        # Use the trusted source temporal fields from the bus dataset
        df["year"] = df["Year"].astype("int64")
        df["month"] = df["Month"].astype("int64")
        df["day_of_week"] = df["Day of Week"]
        df["hour"] = df["Hour of Day"].astype("int64")

        df["temporal_bucket"] = df.apply(
            lambda row: assign_temporal_bucket(row["day_of_week"], row["hour"]),
            axis=1
        )

        # Bus data is organized at Year/Month level, so the congestion-pricing
        # split is based on the available temporal grain.
        df["pre_post_cp"] = np.where(
            df["year"] >= 2025,
            "post",
            "pre"
        )

        df.to_parquet(output_path, index=False)

        print(f"Wrote: {output_path.name} ({len(df):,} rows)")

        del df

Temporal-standardized bus files already exist.
Set REBUILD_BUS_TEMPORAL = True to regenerate them.


In [10]:
# ---------------------------------------------------------------------
# Validate standardized temporal fields for bus data
# ---------------------------------------------------------------------

bus_temporal_files = sorted(BUS_TEMPORAL_DIR.glob("*.parquet"))

temporal_fields = [
    "year",
    "month",
    "day_of_week",
    "hour",
    "temporal_bucket",
    "pre_post_cp",
]

qa_records = []
bucket_counts = []
pre_post_counts = []

for path in bus_temporal_files:
    df = pd.read_parquet(path, columns=temporal_fields)

    for col in temporal_fields:
        qa_records.append({
            "file": path.name,
            "field": col,
            "rows": len(df),
            "missing_count": df[col].isna().sum(),
            "unique_values": df[col].nunique(dropna=True),
        })

    bucket_counts.append(
        df["temporal_bucket"]
        .value_counts(dropna=False)
        .rename_axis("temporal_bucket")
        .reset_index(name="count")
    )

    pre_post_counts.append(
        df["pre_post_cp"]
        .value_counts(dropna=False)
        .rename_axis("pre_post_cp")
        .reset_index(name="count")
    )

    del df

bus_temporal_qa_df = pd.DataFrame(qa_records)

bus_temporal_summary_df = (
    bus_temporal_qa_df
    .groupby("field", as_index=False)
    .agg(
        total_rows=("rows", "sum"),
        missing_count=("missing_count", "sum"),
        min_unique_values=("unique_values", "min"),
        max_unique_values=("unique_values", "max"),
    )
)

bus_temporal_summary_df["missing_rate"] = (
    bus_temporal_summary_df["missing_count"] /
    bus_temporal_summary_df["total_rows"]
)

display(bus_temporal_summary_df)

bus_bucket_distribution_df = (
    pd.concat(bucket_counts, ignore_index=True)
    .groupby("temporal_bucket", dropna=False, as_index=False)["count"]
    .sum()
    .sort_values("temporal_bucket")
)

print("Temporal bucket distribution:")
display(bus_bucket_distribution_df)

bus_pre_post_distribution_df = (
    pd.concat(pre_post_counts, ignore_index=True)
    .groupby("pre_post_cp", dropna=False, as_index=False)["count"]
    .sum()
)

print("Pre/post congestion pricing distribution:")
display(bus_pre_post_distribution_df)

,field,total_rows,missing_count,min_unique_values,max_unique_values,missing_rate
0,day_of_week,18937024,0,7,7,0.0
1,hour,18937024,0,24,24,0.0
2,month,18937024,0,12,12,0.0
3,pre_post_cp,18937024,0,1,1,0.0
4,temporal_bucket,18937024,0,10,10,0.0
5,year,18937024,0,2,2,0.0


Temporal bucket distribution:


,temporal_bucket,count
0,weekday_am_peak,2765464
1,weekday_evening,2341412
2,weekday_midday,3940405
3,weekday_overnight,2219473
4,weekday_pm_peak,2793432
5,weekend_am_peak,861247
6,weekend_evening,864608
7,weekend_midday,1418697
8,weekend_overnight,792046
9,weekend_pm_peak,940240


Pre/post congestion pricing distribution:


,pre_post_cp,count
0,post,7280927
1,pre,11656097


Findings\. Temporal standardization completed successfully for the bus dataset\. All standardized temporal fields were populated without missing values, with full coverage across months, weekdays, hours, and temporal buckets\. The resulting structure aligns with the traffic dataset while preserving the bus dataset's natural observation grain, enabling consistent temporal alignment across mobility systems\.

## 1\.2\.1\.3 Add temporal fields to subway data

The Subway ridership dataset is already staged as quarterly parquet files from the source\-loading step\. In this section, we standardize its temporal representation so it can align with the rest of the multimodal mobility pipeline\. For Subway, the authoritative time field is the ridership timestamp\. We use it to derive the shared temporal fields used across Traffic, Bus, and TLC: \`date\`, \`year\`, \`month\`, \`day\_of\_week\`, \`hour\`, \`temporal\_bucket\`, and \`pre\_post\_cp\`\.This keeps Subway compatible with the project’s canonical analytical grain:

\`Taxi Zone × Date × Temporal Bucket\`

### Test\-derive the temporal fields

Let's define the paths, constants, and helper logic needed to standardize Subway timestamps\. We keep the temporal bucket definitions identical to the rest of the project so Subway can later align cleanly with Traffic, Bus, and TLC at the shared \`Date × Temporal Bucket\` level\.

In [11]:
# ---------------------------------------------------------------------
# 1.2.1.3 Add temporal fields to subway data
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# Input / output paths
# ---------------------------------------------------------------------

SUBWAY_INPUT_DIR = Path("pipeline_data/1.1.5.subway_hourly_ridership_parquet")
SUBWAY_TEMPORAL_OUTPUT_DIR = Path("pipeline_data/1.2.1.subway_ridership_temporal_parquet")

SUBWAY_TEMPORAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Shared temporal constants
# ---------------------------------------------------------------------

CONGESTION_PRICING_START_DATE = pd.to_datetime("2025-01-05").date()

TEMPORAL_BUCKET_ORDER = [
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
]

# ---------------------------------------------------------------------
# Shared temporal bucket logic
# ---------------------------------------------------------------------

def assign_temporal_bucket(day_of_week, hour):
    """
    Assign a standardized temporal bucket using weekday/weekend status
    and hour-of-day daypart logic.

    Buckets are intentionally shared across mobility datasets so Traffic,
    Bus, TLC, Subway, and later Bridge/Tunnel data can be aligned to the
    same temporal representation.
    """
    if day_of_week in ["Saturday", "Sunday"]:
        day_type = "weekend"
    else:
        day_type = "weekday"

    if 0 <= hour <= 5:
        daypart = "overnight"
    elif 6 <= hour <= 9:
        daypart = "am_peak"
    elif 10 <= hour <= 15:
        daypart = "midday"
    elif 16 <= hour <= 19:
        daypart = "pm_peak"
    elif 20 <= hour <= 23:
        daypart = "evening"
    else:
        daypart = "unknown"

    return f"{day_type}_{daypart}"

Let's inspect the staged Subway parquet files before we begin transforming them\. We want to confirm the expected quarterly coverage is present and verify the timestamp field we'll use as the authoritative source for deriving the project's standardized temporal fields\.

In [12]:
# ---------------------------------------------------------------------
# Inspect staged subway parquet files
# ---------------------------------------------------------------------

subway_files = sorted(SUBWAY_INPUT_DIR.glob("*.parquet"))

print(f"Found {len(subway_files):,} parquet files:\n")

for file in subway_files:
    print(file.name)

Found 13 parquet files:

subway_2023_q1.parquet
subway_2023_q2.parquet
subway_2023_q3.parquet
subway_2023_q4.parquet
subway_2024_q1.parquet
subway_2024_q2.parquet
subway_2024_q3.parquet
subway_2024_q4.parquet
subway_2025_q1.parquet
subway_2025_q2.parquet
subway_2025_q3.parquet
subway_2025_q4.parquet
subway_2026_q1.parquet


Before deriving any temporal fields, let's remind ourselves of the schema of a representative Subway file\. This gives us one final sanity check on the timestamp field and confirms the structure matches what we validated during source staging\.

In [13]:
# ---------------------------------------------------------------------
# Inspect subway schema
# ---------------------------------------------------------------------

sample_file = subway_files[0]

schema_df = duckdb.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{sample_file}')
""").df()

schema_df

,column_name,column_type,null,key,default,extra
0,transit_timestamp,TIMESTAMP,YES,None,None,None
1,transit_mode,VARCHAR,YES,None,None,None
2,station_complex_id,VARCHAR,YES,None,None,None
3,station_complex,VARCHAR,YES,None,None,None
4,borough,VARCHAR,YES,None,None,None
5,payment_method,VARCHAR,YES,None,None,None
6,fare_class_category,VARCHAR,YES,None,None,None
7,ridership,DOUBLE,YES,None,None,None
8,transfers,DOUBLE,YES,None,None,None
9,latitude,DOUBLE,YES,None,None,None


Findings: The staged Subway schema looks clean and ready for temporal standardization\. Most importantly, \`transit\_timestamp\` is already stored as a proper \`TIMESTAMP\`, allowing us to derive the project's standard temporal fields directly without additional parsing\. The schema also confirms that ridership data is currently organized at the station\-complex level, with geographic attributes preserved for the spatial standardization step that follows\.

Let's use \`transit\_timestamp\` as the authoritative Subway timestamp and derive the standard temporal fields from it\. This keeps Subway aligned with the rest of the pipeline without adding any subway\-specific time logic\.

In [14]:
# ---------------------------------------------------------------------
# Create standardized temporal fields for one sample subway file
# ---------------------------------------------------------------------

sample_temporal_df = duckdb.sql(f"""
    SELECT
        *,
        CAST(transit_timestamp AS DATE) AS date,
        EXTRACT(year FROM transit_timestamp) AS year,
        EXTRACT(month FROM transit_timestamp) AS month,
        STRFTIME(transit_timestamp, '%A') AS day_of_week,
        EXTRACT(hour FROM transit_timestamp) AS hour
    FROM read_parquet('{sample_file}')
    LIMIT 10
""").df()

sample_temporal_df[
    [
        "transit_timestamp",
        "date",
        "year",
        "month",
        "day_of_week",
        "hour"
    ]
]

,transit_timestamp,date,year,month,day_of_week,hour
0,2023-01-01,2023-01-01,2023,1,Sunday,0
1,2023-01-01,2023-01-01,2023,1,Sunday,0
2,2023-01-01,2023-01-01,2023,1,Sunday,0
3,2023-01-01,2023-01-01,2023,1,Sunday,0
4,2023-01-01,2023-01-01,2023,1,Sunday,0
5,2023-01-01,2023-01-01,2023,1,Sunday,0
6,2023-01-01,2023-01-01,2023,1,Sunday,0
7,2023-01-01,2023-01-01,2023,1,Sunday,0
8,2023-01-01,2023-01-01,2023,1,Sunday,0
9,2023-01-01,2023-01-01,2023,1,Sunday,0


Findings: The temporal fields were derived successfully from \`transit\_timestamp\`\. The sample output shows the expected values for \`date\`, \`year\`, \`month\`, \`day\_of\_week\`, and \`hour\`, confirming that the staged timestamp can serve as the authoritative source for temporal standardization across the Subway dataset\.

Now that the basic timestamp fields look right, let's preview the two remaining standardized fields: \`temporal\_bucket\` and \`pre\_post\_cp\`\. This lets us validate the full temporal logic on a small sample before writing any new Subway parquet files\.

In [15]:
# ---------------------------------------------------------------------
# Preview full temporal-standardization logic on a sample
# ---------------------------------------------------------------------

sample_temporal_full_df = duckdb.sql(f"""
    SELECT
        transit_timestamp,
        CAST(transit_timestamp AS DATE) AS date,
        EXTRACT(year FROM transit_timestamp) AS year,
        EXTRACT(month FROM transit_timestamp) AS month,
        STRFTIME(transit_timestamp, '%A') AS day_of_week,
        EXTRACT(hour FROM transit_timestamp) AS hour,

        CASE
            WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5
                THEN 'weekend_overnight'
            WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9
                THEN 'weekend_am_peak'
            WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15
                THEN 'weekend_midday'
            WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19
                THEN 'weekend_pm_peak'
            WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23
                THEN 'weekend_evening'

            WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5
                THEN 'weekday_overnight'
            WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9
                THEN 'weekday_am_peak'
            WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15
                THEN 'weekday_midday'
            WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19
                THEN 'weekday_pm_peak'
            WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                 AND EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23
                THEN 'weekday_evening'
            ELSE 'unknown'
        END AS temporal_bucket,

        CASE
            WHEN CAST(transit_timestamp AS DATE) < DATE '2025-01-05'
                THEN 'pre'
            ELSE 'post'
        END AS pre_post_cp

    FROM read_parquet('{sample_file}')
    LIMIT 20
""").df()

sample_temporal_full_df

,transit_timestamp,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp
0,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
1,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
2,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
3,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
4,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
5,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
6,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
7,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
8,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre
9,2023-01-01,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre


Findings: The sample output confirms that the full temporal\-standardization logic is working as expected\. The derived fields correctly identify January 1, 2023 as a Sunday at hour 0, assign it to the \`weekend\_overnight\` temporal bucket, and classify it as occurring before the start of congestion pricing\. With the temporal logic validated, we can now generate the standardized Subway parquet files\.

### Write new temporal fields into parquet

Let's generate the temporal\-standardized Subway parquet files only when needed\. Since this is a heavier write step, we use a manual regeneration toggle and also check whether the expected output files already exist\. This lets the notebook safely skip the write when outputs are already present, while still generating them automatically the first time\.

In [16]:
# ---------------------------------------------------------------------
# Generate temporal-standardized subway files
# ---------------------------------------------------------------------

REGENERATE_SUBWAY_TEMPORAL_FILES = False

expected_output_files = [
    SUBWAY_TEMPORAL_OUTPUT_DIR / input_file.name
    for input_file in subway_files
]

missing_output_files = [
    output_file
    for output_file in expected_output_files
    if not output_file.exists()
]

should_generate_subway_temporal_files = (
    REGENERATE_SUBWAY_TEMPORAL_FILES
    or len(missing_output_files) > 0
)

print(f"Expected output files: {len(expected_output_files):,}")
print(f"Existing output files: {len(expected_output_files) - len(missing_output_files):,}")
print(f"Missing output files: {len(missing_output_files):,}")
print(f"Regeneration requested: {REGENERATE_SUBWAY_TEMPORAL_FILES}")

if should_generate_subway_temporal_files:

    print("\nGenerating temporal-standardized Subway parquet files...\n")

    SUBWAY_TEMPORAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for input_file in subway_files:

        output_file = SUBWAY_TEMPORAL_OUTPUT_DIR / input_file.name

        if output_file.exists() and not REGENERATE_SUBWAY_TEMPORAL_FILES:
            print(f"Skipping existing file: {output_file.name}")
            continue

        print(f"Processing {input_file.name}")

        duckdb.sql(f"""
            COPY (
                SELECT
                    *,

                    CAST(transit_timestamp AS DATE) AS date,

                    EXTRACT(year FROM transit_timestamp) AS year,

                    EXTRACT(month FROM transit_timestamp) AS month,

                    STRFTIME(transit_timestamp, '%A') AS day_of_week,

                    EXTRACT(hour FROM transit_timestamp) AS hour,

                    CASE
                        WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5
                            THEN 'weekend_overnight'
                        WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9
                            THEN 'weekend_am_peak'
                        WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15
                            THEN 'weekend_midday'
                        WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19
                            THEN 'weekend_pm_peak'
                        WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23
                            THEN 'weekend_evening'

                        WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5
                            THEN 'weekday_overnight'
                        WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9
                            THEN 'weekday_am_peak'
                        WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15
                            THEN 'weekday_midday'
                        WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19
                            THEN 'weekday_pm_peak'
                        WHEN STRFTIME(transit_timestamp, '%A') NOT IN ('Saturday', 'Sunday')
                             AND EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23
                            THEN 'weekday_evening'
                    END AS temporal_bucket,

                    CASE
                        WHEN CAST(transit_timestamp AS DATE)
                             < DATE '2025-01-05'
                            THEN 'pre'
                        ELSE 'post'
                    END AS pre_post_cp

                FROM read_parquet('{input_file}')
            )
            TO '{output_file}'
            (FORMAT PARQUET)
        """)

    print("\nTemporal-standardized Subway parquet files are ready.")

else:
    print("\nTemporal-standardized Subway parquet files already exist. Skipping generation.")

Expected output files: 13
Existing output files: 13
Missing output files: 0
Regeneration requested: False

Temporal-standardized Subway parquet files already exist. Skipping generation.


### Subway Parquet Validation

Now that the temporal\-standardized Subway files have been written, let's validate only the transformation\-level checks that matter here: row preservation, temporal bucket coverage, and congestion\-pricing period coverage\. The deeper source\-data integrity checks were already handled during Subway staging\.

In [17]:
# ---------------------------------------------------------------------
# Validate row preservation after temporal standardization
# ---------------------------------------------------------------------

subway_input_glob = str(SUBWAY_INPUT_DIR / "*.parquet")
subway_temporal_output_glob = str(SUBWAY_TEMPORAL_OUTPUT_DIR / "*.parquet")

subway_row_preservation_df = duckdb.sql(f"""
    WITH input_counts AS (
        SELECT COUNT(*) AS input_row_count
        FROM read_parquet('{subway_input_glob}')
    ),

    output_counts AS (
        SELECT COUNT(*) AS output_row_count
        FROM read_parquet('{subway_temporal_output_glob}')
    )

    SELECT
        input_row_count,
        output_row_count,
        output_row_count - input_row_count AS row_count_difference
    FROM input_counts
    CROSS JOIN output_counts
""").df()

subway_row_preservation_df

,input_row_count,output_row_count,row_count_difference
0,88319707,88319707,0


Findings\. The temporal\-standardized Subway dataset passed all validation checks\. All 88\.3 million observations were preserved during processing with no row loss or duplication\. The standardized dataset spans the full study period from January 2023 through March 2026, contains all four expected years and all twelve calendar months, and successfully populated all ten temporal buckets and both congestion\-pricing periods\. No null values were introduced in any of the newly derived temporal fields, confirming that temporal standardization completed successfully across the entire Subway dataset\.

Let's confirm that the new temporal bucket field covers the full standardized bucket framework\. This checks the main categorization logic we added in this step\.

In [18]:
# ---------------------------------------------------------------------
# Validate temporal bucket coverage
# ---------------------------------------------------------------------

subway_temporal_bucket_validation_df = duckdb.sql(f"""
    SELECT
        temporal_bucket,
        COUNT(*) AS row_count
    FROM read_parquet('{subway_temporal_output_glob}')
    GROUP BY temporal_bucket
    ORDER BY temporal_bucket
""").df()

subway_temporal_bucket_validation_df

,temporal_bucket,row_count
0,weekday_am_peak,12600063
1,weekday_evening,10323543
2,weekday_midday,18990788
3,weekday_overnight,10831106
4,weekday_pm_peak,12470066
5,weekend_am_peak,4169571
6,weekend_evening,3828983
7,weekend_midday,6706761
8,weekend_overnight,4044099
9,weekend_pm_peak,4354727


Findings: All ten expected temporal buckets are present in the standardized Subway dataset, confirming complete coverage of the project's shared temporal framework\. The distribution of observations appears reasonable, with weekday buckets containing substantially more ridership observations than weekend buckets and midday periods containing the highest overall activity levels\. No unexpected bucket values were observed, indicating that all Subway records were assigned successfully to a valid temporal bucket\.

Let's also confirm that the new congestion\-pricing period field splits observations into the expected pre\- and post\-policy periods\.

In [19]:
# ---------------------------------------------------------------------
# Validate congestion-pricing period coverage
# ---------------------------------------------------------------------

subway_pre_post_validation_df = duckdb.sql(f"""
    SELECT
        pre_post_cp,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(*) AS row_count
    FROM read_parquet('{subway_temporal_output_glob}')
    GROUP BY pre_post_cp
    ORDER BY pre_post_cp
""").df()

subway_pre_post_validation_df

,pre_post_cp,min_date,max_date,row_count
0,post,2025-01-05,2026-03-30,35417907
1,pre,2023-01-01,2025-01-04,52901800


Findings: The congestion\-pricing assignments appear correct and cover the full study period\. The pre\-congestion\-pricing period spans January 1, 2023 through January 4, 2025 and contains approximately 52\.9 million observations, while the post\-congestion\-pricing period spans January 5, 2025 through March 30, 2026 and contains approximately 35\.4 million observations\. The policy boundary aligns exactly with the project's congestion\-pricing start date, confirming that all Subway observations were assigned successfully to the appropriate policy period\.

### Section Summary

Subway temporal standardization is complete\. Standardized temporal fields were derived successfully from \`transit\_timestamp\`, all 88\.3 million observations were preserved, all ten temporal buckets were populated, and congestion\-pricing assignments aligned correctly with the project's policy boundary\. The resulting dataset is now ready for Taxi Zone assignment and spatial standardization in Section 1\.2\.2\.

## 1\.2\.1\.4 Add temporal fields to bridge & tunnel

The Bridges & Tunnels dataset is already reported at an hourly level, but it still needs to be aligned with the project's standard temporal schema before it can be used in downstream multimodal analysis\. In this section, we load the staged Bridges & Tunnels parquet file, convert the source timestamp and date fields into consistent datetime formats, and add the shared temporal fields used across the rest of the pipeline, including year, month, day of week, temporal bucket, and congestion\-pricing period\.

### Load Staged Bridges & Tunnels Dataset

First, define the input and output paths for the Bridges & Tunnels temporal standardization step\. The input is the staged parquet created during the loading notebook, while the output will be the temporally standardized parquet used in later pipeline stages\.

In [20]:
# ---------------------------------------------------------------------
# 1.2.1.4 Add temporal fields to bridge & tunnel
# ---------------------------------------------------------------------

REBUILD_BRIDGE_TUNNEL_TEMPORAL = False

BRIDGE_TUNNEL_STAGED_PARQUET_PATH = (
    PIPELINE_DATA_DIR /
    "1.1.3.bridge_tunnel_hourly_crossings" /
    "bridge_tunnel_hourly_crossings_staged.parquet"
)

BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH = (
    PIPELINE_DATA_DIR /
    "1.2.1.bridge_tunnel_temporal.parquet"
)

Next, load the staged Bridges & Tunnels parquet file and confirm that the dataset is available for temporal standardization\.

In [21]:
bridge_tunnel_temporal = pd.read_parquet(
    BRIDGE_TUNNEL_STAGED_PARQUET_PATH
)

print("Bridge & Tunnel staged shape:", bridge_tunnel_temporal.shape)

display(bridge_tunnel_temporal.head())

Bridge & Tunnel staged shape: (5941375, 11)


,transit_timestamp,date,hour,facility_id,facility,direction,payment_method,vehicle_class,vehicle_class_description,vehicle_class_category,traffic_count
0,01/01/2023 12:00:00 AM,01/01/2023,0,27,Queens Midtown Tunnel,Westbound to Manhattan,Tolls by Mail,9,motorcycle,Motorcycle,2
1,01/01/2023 12:00:00 AM,01/01/2023,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,Tolls by Mail,1,2-axle passenger car,Car,83
2,01/01/2023 12:00:00 AM,01/01/2023,0,29,Throgs Neck Bridge,Southbound to Queens,E-ZPass,6,3-axle truck,Truck,1
3,01/01/2023 12:00:00 AM,01/01/2023,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,E-ZPass,1,2-axle passenger car,Car,404
4,01/01/2023 12:00:00 AM,01/01/2023,0,30,Verrazzano - Narrows Bridge,Eastbound to Brooklyn,E-ZPass,8,5-axle truck,Truck,3


Finally, inspect the staged schema before adding new temporal fields\. This helps confirm that the expected timestamp, date, hour, facility, vehicle classification, payment method, and traffic count fields are present before transformation\.

In [22]:
bridge_tunnel_temporal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5941375 entries, 0 to 5941374
Data columns (total 11 columns):
 #   Column                     Dtype 
---  ------                     ----- 
 0   transit_timestamp          object
 1   date                       object
 2   hour                       int64 
 3   facility_id                int64 
 4   facility                   object
 5   direction                  object
 6   payment_method             object
 7   vehicle_class              int64 
 8   vehicle_class_description  object
 9   vehicle_class_category     object
 10  traffic_count              int64 
dtypes: int64(4), object(7)
memory usage: 498.6+ MB


### Generate Standard Temporal Features

Next, we add the standard temporal fields used across the congestion\-pricing pipeline\. Although the Bridges & Tunnels dataset is already reported at an hourly level, we still need to normalize the timestamp and date fields and generate the shared temporal dimensions used by the other mobility datasets\. This includes year, month, day of week, weekday/weekend daypart buckets, and the pre/post congestion\-pricing indicator\.

First, convert the source timestamp and date fields into proper datetime objects\. This ensures that downstream temporal fields are derived from consistent datetime values rather than string representations\.

In [23]:
bridge_tunnel_temporal["transit_timestamp"] = pd.to_datetime(
    bridge_tunnel_temporal["transit_timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

bridge_tunnel_temporal["date"] = pd.to_datetime(
    bridge_tunnel_temporal["date"],
    format="%m/%d/%Y",
    errors="coerce"
).dt.date

Next, derive the standard calendar fields used throughout the project\. These fields make the Bridges & Tunnels dataset consistent with the temporal schema already applied to Traffic, Bus, Subway, and TLC data\.

In [24]:
bridge_tunnel_temporal["year"] = pd.to_datetime(
    bridge_tunnel_temporal["date"]
).dt.year

bridge_tunnel_temporal["month"] = pd.to_datetime(
    bridge_tunnel_temporal["date"]
).dt.month

bridge_tunnel_temporal["day_of_week"] = pd.to_datetime(
    bridge_tunnel_temporal["date"]
).dt.day_name()

bridge_tunnel_temporal["hour"] = bridge_tunnel_temporal["hour"].astype(int)

display(bridge_tunnel_temporal.head())

,transit_timestamp,date,hour,facility_id,facility,direction,payment_method,vehicle_class,vehicle_class_description,vehicle_class_category,traffic_count,year,month,day_of_week
0,2023-01-01,2023-01-01,0,27,Queens Midtown Tunnel,Westbound to Manhattan,Tolls by Mail,9,motorcycle,Motorcycle,2,2023,1,Sunday
1,2023-01-01,2023-01-01,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,Tolls by Mail,1,2-axle passenger car,Car,83,2023,1,Sunday
2,2023-01-01,2023-01-01,0,29,Throgs Neck Bridge,Southbound to Queens,E-ZPass,6,3-axle truck,Truck,1,2023,1,Sunday
3,2023-01-01,2023-01-01,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,E-ZPass,1,2-axle passenger car,Car,404,2023,1,Sunday
4,2023-01-01,2023-01-01,0,30,Verrazzano - Narrows Bridge,Eastbound to Brooklyn,E-ZPass,8,5-axle truck,Truck,3,2023,1,Sunday


Then, assign each hourly observation to the project's standard temporal bucket definitions\. These buckets combine weekday/weekend behavior with daypart structure so that each mobility dataset can be compared on the same time\-of\-week basis\.

In [25]:
def assign_temporal_bucket(day_of_week, hour):
    is_weekend = day_of_week in ["Saturday", "Sunday"]

    if is_weekend:
        if 0 <= hour < 6:
            return "weekend_overnight"
        elif 6 <= hour < 10:
            return "weekend_am_peak"
        elif 10 <= hour < 16:
            return "weekend_midday"
        elif 16 <= hour < 20:
            return "weekend_pm_peak"
        else:
            return "weekend_evening"

    else:
        if 0 <= hour < 6:
            return "weekday_overnight"
        elif 6 <= hour < 10:
            return "weekday_am_peak"
        elif 10 <= hour < 16:
            return "weekday_midday"
        elif 16 <= hour < 20:
            return "weekday_pm_peak"
        else:
            return "weekday_evening"


bridge_tunnel_temporal["temporal_bucket"] = bridge_tunnel_temporal.apply(
    lambda row: assign_temporal_bucket(
        row["day_of_week"],
        row["hour"]
    ),
    axis=1
)

bridge_tunnel_temporal["temporal_bucket"].value_counts()

temporal_bucket
weekday_midday       1220613
weekday_overnight     918220
weekday_am_peak       812941
weekday_pm_peak       756877
weekday_evening       687068
weekend_midday        420873
weekend_overnight     337173
weekend_pm_peak       269934
weekend_am_peak       266543
weekend_evening       251133
Name: count, dtype: int64

Finally, add the pre/post congestion\-pricing indicator using the same project cutoff date applied to the other mobility datasets\. This creates the core treatment\-period flag needed for downstream comparison and modeling\.

In [26]:
CONGESTION_PRICING_START_DATE = pd.to_datetime("2025-01-05").date()

bridge_tunnel_temporal["pre_post_cp"] = bridge_tunnel_temporal["date"].apply(
    lambda x: "post_cp" if x >= CONGESTION_PRICING_START_DATE else "pre_cp"
)

bridge_tunnel_temporal[[
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "temporal_bucket",
    "pre_post_cp"
]].head()

,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp
0,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre_cp
1,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre_cp
2,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre_cp
3,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre_cp
4,2023-01-01,2023,1,Sunday,0,weekend_overnight,pre_cp


The standard temporal fields were successfully added to the Bridges & Tunnels dataset, bringing it into alignment with the temporal schema used across the other mobility sources\. The resulting dataset now contains consistent calendar attributes, temporal buckets, and congestion\-pricing period indicators that will support downstream integration, aggregation, and comparative analysis across modalities\.

### Validate Temporal Coverage

Let's now validate their coverage and completeness\. This QA step confirms that all records were successfully assigned the expected temporal dimensions and that the resulting values align with the dataset's observed date range and reporting structure\. Consistent temporal coverage is essential for ensuring that the Bridges & Tunnels dataset can be integrated reliably with the project's other mobility sources\.

First, check whether any of the standardized temporal fields contain missing values\. This confirms that timestamp parsing, calendar feature generation, temporal bucket assignment, and congestion\-pricing period assignment completed successfully for every record\.

In [27]:
# ---------------------------------------------------------------------
# Validate Temporal Coverage
# ---------------------------------------------------------------------

bridge_tunnel_temporal_fields = [
    "transit_timestamp",
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "temporal_bucket",
    "pre_post_cp"
]

bridge_tunnel_temporal_missingness_df = (
    bridge_tunnel_temporal[bridge_tunnel_temporal_fields]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={
        "index": "field",
        0: "missing_count"
    })
)

bridge_tunnel_temporal_missingness_df["missing_pct"] = (
    bridge_tunnel_temporal_missingness_df["missing_count"] /
    len(bridge_tunnel_temporal)
)

display(bridge_tunnel_temporal_missingness_df)

,field,missing_count,missing_pct
0,transit_timestamp,0,0.0
1,date,0,0.0
2,year,0,0.0
3,month,0,0.0
4,day_of_week,0,0.0
5,hour,0,0.0
6,temporal_bucket,0,0.0
7,pre_post_cp,0,0.0


Next, summarize the range and cardinality of the standardized temporal fields\. This gives us a compact view of the dataset's date range, hourly coverage, temporal bucket coverage, and pre/post congestion\-pricing coverage\.

In [28]:
bridge_tunnel_temporal_coverage_df = pd.DataFrame({
    "field": [
        "transit_timestamp",
        "date",
        "year",
        "month",
        "day_of_week",
        "hour",
        "temporal_bucket",
        "pre_post_cp"
    ],
    "min_value": [
        bridge_tunnel_temporal["transit_timestamp"].min(),
        bridge_tunnel_temporal["date"].min(),
        bridge_tunnel_temporal["year"].min(),
        bridge_tunnel_temporal["month"].min(),
        bridge_tunnel_temporal["day_of_week"].min(),
        bridge_tunnel_temporal["hour"].min(),
        bridge_tunnel_temporal["temporal_bucket"].min(),
        bridge_tunnel_temporal["pre_post_cp"].min()
    ],
    "max_value": [
        bridge_tunnel_temporal["transit_timestamp"].max(),
        bridge_tunnel_temporal["date"].max(),
        bridge_tunnel_temporal["year"].max(),
        bridge_tunnel_temporal["month"].max(),
        bridge_tunnel_temporal["day_of_week"].max(),
        bridge_tunnel_temporal["hour"].max(),
        bridge_tunnel_temporal["temporal_bucket"].max(),
        bridge_tunnel_temporal["pre_post_cp"].max()
    ],
    "distinct_values": [
        bridge_tunnel_temporal["transit_timestamp"].nunique(),
        bridge_tunnel_temporal["date"].nunique(),
        bridge_tunnel_temporal["year"].nunique(),
        bridge_tunnel_temporal["month"].nunique(),
        bridge_tunnel_temporal["day_of_week"].nunique(),
        bridge_tunnel_temporal["hour"].nunique(),
        bridge_tunnel_temporal["temporal_bucket"].nunique(),
        bridge_tunnel_temporal["pre_post_cp"].nunique()
    ]
})

display(bridge_tunnel_temporal_coverage_df)

,field,min_value,max_value,distinct_values
0,transit_timestamp,2023-01-01 00:00:00,2026-05-12 23:00:00,29468
1,date,2023-01-01,2026-05-12,1228
2,year,2023,2026,4
3,month,1,12,12
4,day_of_week,Friday,Wednesday,7
5,hour,0,23,24
6,temporal_bucket,weekday_am_peak,weekend_pm_peak,10
7,pre_post_cp,post_cp,pre_cp,2


Finally, review the distribution of records across the derived temporal buckets and congestion\-pricing periods\. These counts are based on source observations rather than traffic volumes, so they are mainly used to verify that the new fields were assigned as expected\.

In [29]:
bridge_tunnel_temporal_bucket_counts_df = (
    bridge_tunnel_temporal["temporal_bucket"]
    .value_counts()
    .rename_axis("temporal_bucket")
    .reset_index(name="observation_count")
)

display(bridge_tunnel_temporal_bucket_counts_df)

,temporal_bucket,observation_count
0,weekday_midday,1220613
1,weekday_overnight,918220
2,weekday_am_peak,812941
3,weekday_pm_peak,756877
4,weekday_evening,687068
5,weekend_midday,420873
6,weekend_overnight,337173
7,weekend_pm_peak,269934
8,weekend_am_peak,266543
9,weekend_evening,251133


In [30]:
bridge_tunnel_pre_post_counts_df = (
    bridge_tunnel_temporal["pre_post_cp"]
    .value_counts()
    .rename_axis("pre_post_cp")
    .reset_index(name="observation_count")
)

display(bridge_tunnel_pre_post_counts_df)

,pre_post_cp,observation_count
0,pre_cp,3602730
1,post_cp,2338645


Findings: All standardized temporal fields exhibited complete coverage with no missing values detected\. The dataset spans January 2023 through May 2026 and contains the full expected range of years, months, days of week, hours, temporal buckets, and congestion\-pricing periods\. All temporal buckets and both pre\- and post\-congestion pricing periods were populated successfully, indicating that the temporal standardization process was applied consistently across the full dataset\.

### Save Output

With the standard temporal fields added and validated, the final step is to save the Bridges & Tunnels dataset as a temporally standardized parquet file\. This output will serve as the canonical Bridge & Tunnel input for later harmonization and modeling steps, where it can be used as a global mobility signal joined by date and temporal bucket\.

First, write the temporally standardized Bridges & Tunnels dataset to the pipeline output directory\.

In [31]:
# ---------------------------------------------------------------------
# Save Output
# ---------------------------------------------------------------------

bridge_tunnel_temporal.to_parquet(
    BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH,
    index=False
)

print(
    f"Saved temporally standardized Bridges & Tunnels dataset to:\n"
    f"{BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH}"
)

Saved temporally standardized Bridges & Tunnels dataset to:
pipeline_data/1.2.1.bridge_tunnel_temporal.parquet


Next, reload the saved parquet file and confirm that the output preserves the expected row and column structure\.

In [32]:
bridge_tunnel_temporal_check = pd.read_parquet(
    BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH
)

print("Original shape:", bridge_tunnel_temporal.shape)
print("Reloaded shape:", bridge_tunnel_temporal_check.shape)

display(bridge_tunnel_temporal_check.head())

Original shape: (5941375, 16)
Reloaded shape: (5941375, 16)


,transit_timestamp,date,hour,facility_id,facility,direction,payment_method,vehicle_class,vehicle_class_description,vehicle_class_category,traffic_count,year,month,day_of_week,temporal_bucket,pre_post_cp
0,2023-01-01,2023-01-01,0,27,Queens Midtown Tunnel,Westbound to Manhattan,Tolls by Mail,9,motorcycle,Motorcycle,2,2023,1,Sunday,weekend_overnight,pre_cp
1,2023-01-01,2023-01-01,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,Tolls by Mail,1,2-axle passenger car,Car,83,2023,1,Sunday,weekend_overnight,pre_cp
2,2023-01-01,2023-01-01,0,29,Throgs Neck Bridge,Southbound to Queens,E-ZPass,6,3-axle truck,Truck,1,2023,1,Sunday,weekend_overnight,pre_cp
3,2023-01-01,2023-01-01,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,E-ZPass,1,2-axle passenger car,Car,404,2023,1,Sunday,weekend_overnight,pre_cp
4,2023-01-01,2023-01-01,0,30,Verrazzano - Narrows Bridge,Eastbound to Brooklyn,E-ZPass,8,5-axle truck,Truck,3,2023,1,Sunday,weekend_overnight,pre_cp


Finally, compare the original and reloaded row counts to confirm that no records were dropped during the save process\.

In [33]:
row_count_difference = (
    len(bridge_tunnel_temporal) -
    len(bridge_tunnel_temporal_check)
)

print(f"Row count difference: {row_count_difference:,}")

Row count difference: 0


The temporally standardized Bridges & Tunnels dataset was successfully written to parquet and reloaded without issue\. Row counts matched exactly between the source and output datasets, indicating that no records were lost during the save process\. The resulting dataset preserves the full set of bridge and tunnel crossing observations while providing a consistent temporal schema that can be used alongside the project's other mobility datasets\.

## Close

In this notebook, we standardized the temporal representations used across the Traffic, Bus, and Subway mobility datasets by deriving consistent date and time fields, assigning observations to the project's shared weekday/weekend daypart buckets, and creating pre\- and post\-congestion\-pricing indicators\. This established a common temporal framework that will support downstream spatial harmonization, multimodal integration, and mobility\-state analysis\. Taxi, Green Taxi, and FHVHV temporal standardization were handled separately in Section 1\.2\.3 because those datasets required significant aggregation and restructuring to remain computationally manageable at scale\. With temporal standardization now complete across all major mobility sources, the next step is spatial standardization, where Traffic, Bus, and Subway observations will be aligned to the project's canonical Taxi Zone geography\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>